In [3]:
import numpy as np

# 替换为你的文件路径
file_path = '../data/course/course_rqvae_codes.npy'

# 加载数据
data = np.load(file_path, allow_pickle=True)

# 查看基本信息
print(data[:10])
print(f"数据类型: {type(data)}")
print(f"数组维度 (Shape): {data.shape}")
print(f"元素类型 (Dtype): {data.dtype}")

[[6 1 2 0]
 [2 6 2 0]
 [2 6 4 0]
 [2 6 4 1]
 [2 6 4 2]
 [4 6 1 0]
 [4 5 6 0]
 [2 6 0 0]
 [7 6 2 0]
 [0 0 3 0]]
数据类型: <class 'numpy.ndarray'>
数组维度 (Shape): (707, 4)
元素类型 (Dtype): int32


In [14]:
import h5py
rec_data = []
with h5py.File('../data/user_item_interact.h5', 'r') as f1:
    user_ids = f1['user_id'][:]          # 提取所有 user_id
    user_profile = f1['user_profile'].asstr(encoding='utf-8')[:] # 提取所有 user_profile
    item_lists = f1['item_id_list'][:]    # 提取所有变长数组
    rec_data = list(zip(user_ids, user_profile, item_lists))
# print(rec_data[:5])
print(len(item_lists))

199199


In [16]:
train_list = []
val_list = []
test_list = []

for user_seq in item_lists:
    seq_len = len(user_seq)
    
    # 基础过滤：至少有2个行为才能产生 1个输入 + 1个目标
    if seq_len < 2:
        continue
    
    # --- 1. 测试集 (固定取最后一个) ---
    test_list.append({
        'history': user_seq[:-1].tolist(),
        'target': user_seq[-1]
    })
    
    # --- 2. 验证集 (固定取倒数第二个，前提是序列长度够) ---
    if seq_len >= 3:
        val_list.append({
            'history': user_seq[:-2].tolist(),
            'target': user_seq[-2]
        })
    
    # --- 3. 训练集 (滑动窗口：从第2个商品开始作为 target) ---
    # 我们遍历序列，让每一个位置 i (从1到倒数第2个) 都尝试做一次 target
    # 限制结束位置为 seq_len-2 或 seq_len-1 取决于你是否想让训练集包含验证集的 target
    # 这里我们严格一点，训练集只用到验证集 target 之前的位置
    end_idx = seq_len - 2 if seq_len >= 3 else seq_len - 1
    
    for i in range(1, end_idx + 1):
        train_list.append({
            'history': user_seq[:i].tolist(),
            'target': user_seq[i]
        })

print(f"训练样本数: {len(train_list)}")
print(f"验证样本数: {len(val_list)}")
print(f"测试样本数: {len(test_list)}")

训练样本数: 388131
验证样本数: 95423
测试样本数: 199199


In [18]:
import h5py
import numpy as np

def save_single_h5(file_name, data_list):
    """
    将包含 {'history': [...], 'target': id} 的列表保存为单个 H5 文件
    """
    with h5py.File(file_name, 'w') as f:
        # 提取 target (假设是单个整数 ID)
        targets = np.array([item['target'] for item in data_list], dtype='int32')
        f.create_dataset("target", data=targets)
        
        # 提取 history (变长序列)
        histories = [item['history'] for item in data_list]
        
        # 定义 HDF5 的变长整数类型
        vlen_int = h5py.special_dtype(vlen=np.dtype('int32'))
        
        # 创建 history 数据集
        ds_hist = f.create_dataset("history", (len(histories),), dtype=vlen_int)
        
        # 逐行写入变长历史
        for i, h in enumerate(histories):
            ds_hist[i] = np.array(h, dtype='int32')
            
    print(f"成功保存至: {file_name} | 样本数: {len(data_list)}")

# 分别保存三个文件
save_single_h5("../data/tiger/train_dataset.h5", train_list)
save_single_h5("../data/tiger/val_dataset.h5", val_list)
save_single_h5("../data/tiger/test_dataset.h5", test_list)

成功保存至: ../data/tiger/train_dataset.h5 | 样本数: 388131
成功保存至: ../data/tiger/val_dataset.h5 | 样本数: 95423
成功保存至: ../data/tiger/test_dataset.h5 | 样本数: 199199


In [26]:
with h5py.File('../data/tiger/train_dataset.h5', 'r') as f:
        # user_ids = f['user_id'][:]          # 提取所有 user_id
        # user_profile = f['user_profile'].asstr(encoding='utf-8')[:] # 提取所有 user_profile
        item_history = f['history'][:]
        item_target = f['target'][:]
        # print(item_history[:10])
        print(item_target[:108])


[ 63  28  52 140 190 243  28  63 140  25 492  47  72 107  50 405 167  42
  84 405 179  28 100  77  29  32 100  67  70  36  63 100 188  58  21 152
 368  50 140 119 358 187 123 104 100 358 560 556 552 652  24 152   7 140
  23  67   5  96  80   8 251  96 172 357 686  28 660  80  28  50 217  58
  30 205 102  28  63 100 477  58  96  10 212 378 172 256 143 179  63  66
 601 401 162 190  98 169  78 286 235 216 287 234 140  28  58 514 401 559]
